In [1]:
import os
import sys
import numpy as np
sys.path.append("../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune_head import (
    head_importance_prunning, compute_heads_importance
)

In [3]:
name = "IMDB"
device = torch.device("cuda:0")
checkpoint = None
batch_size = 16
num_workers = 4
num_samples = 128
ci_ratio = 0.4
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [4]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'textattack/bert-base-uncased-imdb', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'IMDB', 'num_labels': 2, 'cache_dir': 'Models'}
The model textattack/bert-base-uncased-imdb is loaded.


In [5]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name, batch_size=16, num_workers=48
)

Loading cached dataset IMDB.
train.pkl is loaded from cache.
valid.pkl is loaded from cache.
test.pkl is loaded from cache.
The dataset IMDB is loaded
{'dataset_name': 'IMDB', 'path': 'imdb', 'config_name': 'plain_text', 'text_column': 'text', 'label_column': 'label', 'cache_dir': 'Datasets/IMDB', 'task_type': 'classification'}


In [6]:
train = copy.deepcopy(train_dataloader)

In [7]:
positive_samples = SamplingDataset(
    train, 0, num_samples, num_labels, True, 4, device=device, resample=False, seed=seed
)

In [8]:
import math
import torch.nn as nn

self_outputs = []
forward_hooks = []
backward_hooks = []

def backward_hook(module, grad_input, grad_output):
    grad = grad_output[0].cpu()
    self_outputs.append(grad)
    return None
        
def forward_hook(module, input, output):
    hidden_states = input[0]
    attention_mask = input[1]
    mixed_query_layer = module.query(hidden_states)
    mixed_key_layer = module.key(hidden_states)
    mixed_value_layer = module.value(hidden_states)
    
    query_layer = module.transpose_for_scores(mixed_query_layer)
    key_layer = module.transpose_for_scores(mixed_key_layer)
    value_layer = module.transpose_for_scores(mixed_value_layer)

    attention_scores = torch.matmul(
        query_layer, key_layer.transpose(-1, -2))
    attention_scores = attention_scores / math.sqrt(module.attention_head_size)

    if attention_mask is not None:
        attention_scores = attention_scores + attention_mask

    attention_probs = nn.Softmax(dim=-1)(attention_scores)
    attention_probs = module.dropout(attention_probs)

    context_layer = torch.matmul(attention_probs, value_layer)
    
    context_layer2 = context_layer.permute(0, 2, 1, 3).contiguous()
    
    new_context_layer_shape = context_layer2.size()[:-2] + (module.all_head_size,)
    context_layer3 = context_layer2.view(*new_context_layer_shape)
    
    output[0].requires_grad_(True)
    output[0].retain_grad()
    self_outputs.append({
        "context_layer_val": context_layer.detach().cpu(),
        "context_layer": output[0].detach().cpu(),
        "shape1": context_layer.shape,
        "shape2": context_layer2.shape,
        "shape3": context_layer3.shape
    })
    return output
        
def register_hooks(model):
    for layer in range(model.bert.config.num_hidden_layers):
        self_att = model.bert.encoder.layer[layer].attention.self
        hook = self_att.register_forward_hook(forward_hook)
        forward_hooks.append(hook)

def register_backward_hooks(model):
    for layer in range(model.bert.config.num_hidden_layers):
        self_att = model.bert.encoder.layer[layer].attention.self
        hook = self_att.register_full_backward_hook(backward_hook)
        backward_hooks.append(hook)

def calculate_head_importance(
        model,
        dataloader,
        device=None,
        normalize_scores_by_layer=True,
        subset_size=1.0,
):
    """Calculate head importance scores"""
    
    # Disable dropout
    model.eval()
    # Device
    device = device or next(model.parameters()).device

    # Head importance tensor
    n_layers = model.bert.config.num_hidden_layers
    n_heads = model.bert.config.num_attention_heads
    head_importance = torch.zeros(n_layers, n_heads)
    tot_tokens = 0

    for step, batch in enumerate(dataloader):
        self_outputs.clear()
        self_grads.clear()
        
        register_hooks(model)
        batch_size = batch["input_ids"].shape[0]
        input_ids = batch["input_ids"].to(device)
        input_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        model.zero_grad()

        # Compute gradients
        output = model(input_ids, labels=labels, attention_mask=input_mask)

        for hook in forward_hooks:
            hook.remove()

        register_backward_hooks(model)
        loss = output.loss
        loss.backward()
        
        for hook in backward_hooks:
            hook.remove()
            
        for layer in range(n_layers):
            grads = self_outputs[layer]
            ctx_val = self_outputs[layer]["context_layer_val"]
            shape2 = self_outputs[layer]["shape2"]
            
            grads = grads.view(shape2).cpu()
            grads = grads.permute(0, 2, 1, 3)
        
            dot = torch.einsum("bhli,bhli->bhl", [grads, ctx_val])
            head_importance[layer] += dot.abs().sum(-1).sum(0).detach()
        
            del grads, ctx, ctx_val, dot

        tot_tokens += input_mask.float().detach().sum().cpu().data

In [9]:
hi1 = calculate_head_importance(
    model, 
    train_dataloader, 
    device="cuda", 
    normalize_scores_by_layer=True,
    subset_size=1.0,
)

NameError: name 'self_grads' is not defined